In [ ]:
!pip install pygwalker

In [ ]:
!pip install bokeh

In [ ]:
# IMPORTAR LIBRERÍAS

from bokeh.plotting import figure, show
from bokeh.io import output_notebook
from bokeh.models import ColumnDataSource
from math import pi
from bokeh.transform import cumsum
from bokeh.palettes import Category10
from bokeh.plotting import figure, show

import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import pygwalker as pyg

In [ ]:
# CONECTAR CON LA CUENTA DE GOOGLE DRIVE CON EL ENTORNO DE PYTHON

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# LEER ARCHIVO CSV

df_csv = pd.read_csv('/content/drive/MyDrive/data/frutas_5000_datos.csv')

# Mostrar primeras filas del CSV
print("DATOS CSV")
print(df_csv.head())


In [ ]:
# CREAR COLUMNA SurveyID
# Se crea una columna identificadora para poder unir el CSV con SQLite

df_csv["ID"] = range(1, 5001)


In [ ]:
# CONECTAR A SQLITE

conexion = sqlite3.connect('/content/drive/MyDrive/data/personas_frutas.sqlite')

cursor = conexion.cursor()


In [ ]:
# LEER TABLA SQLITE

query = "SELECT * FROM personas_consumo_frutas"

df_sql = pd.read_sql_query(query, conexion)

# Mostrar primeras filas
print("\nDATOS SQLITE")
print(df_sql.head())


In [ ]:
# REALIZA LA UNIÓN .CSV Y .SQLITE

# Se unen ambos DataFrames usando SurveyID; how='inner' Une únicamente coincidencias

df_unido = pd.merge(
    df_csv,
    df_sql,
    on="IDFruta",
    how="inner"
)

# Mostrar resultado
print("\nDATAFRAME UNIDO")
print(df_unido.head())

# GUARDAR EL MERGE EN SQLITE

df_unido.to_sql(
    "merge_frutas_personas",
    conexion,
    if_exists="replace",
    index=False
)

print("\nMERGE GUARDADO EN SQLITE")



In [ ]:
# GRÁFICO DE BARRAS DE CANTIDAD DE PERSONAS POR FRUTA FAVORITA

frutas_favoritas = df_unido["FrutaFavorita"].value_counts()

plt.figure(figsize=(10,6))

plt.bar(
    frutas_favoritas.index,
    frutas_favoritas.values
)

plt.title("Cantidad de personas por fruta favorita")

plt.xlabel("Frutas")

plt.ylabel("Cantidad de personas")

plt.xticks(rotation=45)

plt.show()


In [ ]:
# GRÁFICO CIRCULAR DE PORCENTAJE DE CONSUMO DE FRUTAS

consumo = df_unido["FrutaFavorita"].value_counts()

plt.figure(figsize=(7,7))

plt.pie(
    consumo.values,
    labels=consumo.index,
    autopct="%1.1f%%"
)

plt.title("Porcentaje de consumo de frutas")

plt.show()

In [ ]:
# CONTAR FRUTASCon Gráficos de BOKEH

conteo = df_csv["Fruta"].value_counts()

frutas = conteo.index.tolist()

cantidades = conteo.values.tolist()

# CREAR GRÁFICO
p = figure(
    x_range=frutas,
    height=400,
    width=700,
    title="Cantidad de registros por fruta",
    toolbar_location=None
)

#Barras
p.vbar(
    x=frutas,
    top=cantidades,
    width=0.5
)

# ETIQUETAS
p.xaxis.axis_label = "Frutas"

p.yaxis.axis_label = "Cantidad"

show(p)

In [ ]:
# MOSTRAR GRÁFICO
output_notebook()

# Agrupar datos
datos = df_csv.groupby("Fruta")["Cantidad"].mean().reset_index()

# Crear gráfico
p = figure(
    x_range=datos["Fruta"],
    height=400,
    title="Promedio de cantidad por fruta"
)

# Crear burbujas
p.scatter(
    x=datos["Fruta"],
    y=datos["Cantidad"],
    size=datos["Cantidad"],
    alpha=0.6
)

# Mostrar gráfico
show(p)

In [ ]:
# GRÁFICO CON PYGWALKER

 pyg.walk(df_csv)

In [ ]:
# CONSULTAR TABLAS GUARDADAS

query_tablas = """
SELECT name
FROM sqlite_master
WHERE type='table';
"""

tablas = pd.read_sql_query(query_tablas, conexion)

print("\nTABLAS EN SQLITE")
print(tablas)


In [ ]:
# CONSULTAR DATOS DEL MERGE

query_merge = """
SELECT *
FROM merge_frutas_personas
LIMIT 5;
"""

resultado_merge = pd.read_sql_query(
    query_merge,
    conexion
)

print("\nDATOS DEL MERGE EN SQLITE")
print(resultado_merge)

In [ ]:
# EXPORTAR NUEVA BASE SQLITE

print("\nBASE DE DATOS SQLITE ACTUALIZADA")

# conexion.close()